In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import truncnorm
import matplotlib.pyplot as plt
import seaborn as sns

# Set seed for reproducibility 
np.random.seed(42)
N_SAMPLES = 1000

print(f"Initializing Data Engine: Generating {N_SAMPLES} observations...")

# =================================================================
# STEP 1: HELPER FUNCTIONS (Fuzzy Logic)
# =================================================================

def get_truncated_normal(mean=0, sd=1, low=0, upp=10):
    """
    Generates a normal distribution capped at specific boundaries.
    Crucial for representing 'Membership Functions' within a fixed range.
    """
    return truncnorm((low - mean) / sd, (upp - mean) / sd, loc=mean, scale=sd)

# =================================================================
# STEP 2: GENERATING PREDICTORS (Inputs))
# =================================================================

# 1. Client Technical Literacy (0.0 = Novice, 1.0 = Expert)
literacy_gen = get_truncated_normal(mean=0.5, sd=0.25, low=0, upp=1)
client_tech_literacy = literacy_gen.rvs(N_SAMPLES)

# 2. Ownership Clarity (0.0 = Ambiguous/Personality-driven, 1.0 = Process-driven)
# REVISION: Upgraded from binary 'owner_assigned' to continuous membership function.
clarity_gen = get_truncated_normal(mean=0.4, sd=0.3, low=0, upp=1)
ownership_clarity = clarity_gen.rvs(N_SAMPLES)

# 3. Scope Lock Day (Integer 1-30)
# REVISION: Implementing Risk Factor 4 (Scope Elasticity)
scope_lock_day = np.random.poisson(14, N_SAMPLES)
scope_lock_day = np.clip(scope_lock_day, 1, 30)

# 4. Asset Depth (0.0 = Sparse, 1.0 = Comprehensive)
# Logic: Strong correlation with Ownership Clarity (process drives documentation).
asset_depth = []
for clarity in ownership_clarity:
    mean_depth = clarity # High clarity correlates with better documentation
    depth = get_truncated_normal(mean=mean_depth, sd=0.15, low=0, upp=1).rvs()
    asset_depth.append(depth)
asset_depth = np.array(asset_depth)

# =================================================================
# STEP 3: THE FUZZY INFERENCE RULES (Interaction Logic)
# =================================================================

# 1. Repeat Query Rate (Documentation effect on support volume)
# Logic: Lower literacy AND lower assets = Exponentially more friction.
noise = np.random.normal(0, 0.05, N_SAMPLES)
repeat_query_rate = (1 - client_tech_literacy) * (1 - asset_depth) + noise
repeat_query_rate = np.clip(repeat_query_rate, 0, 1)

# 2. Founder Involvement (founder time cost)
# REVISION: Probabilistic function of CONTINUOUS ownership clarity.
# Using an exponential decay to model how clarity eliminates founder dependency.
founder_involved = []
for clarity in ownership_clarity:
    prob = np.exp(-4 * clarity) # High clarity = near-zero founder need
    founder_involved.append(np.random.choice([0, 1], p=[1-np.clip(prob, 0, 1), np.clip(prob, 0, 1)]))
founder_involved = np.array(founder_involved)

# 3. Time to First Value (TTFV) - The Activation Metric
# REVISION: Implementing the 1.3x - 1.5x Scope Penalty for late locks (> Day 14)
base_ttfv = 20
scope_penalty = np.where(scope_lock_day > 14, np.random.uniform(1.3, 1.5, N_SAMPLES), 1.0)

ttfv_days = (
    (base_ttfv + 
    (1 - client_tech_literacy) * 20 + 
    (1 - asset_depth) * 25 + 
    (founder_involved * 10)) * scope_penalty + 
    np.random.poisson(5, N_SAMPLES)
)

# 4. Satisfaction Score (1.0 to 5.0)
cx_satisfaction = 5 - (repeat_query_rate * 3) - (ttfv_days / 60) + np.random.normal(0, 0.2, N_SAMPLES)
cx_satisfaction = np.clip(cx_satisfaction, 1, 5)

# =================================================================
# STEP 4: CONSOLIDATION & EXPORT
# =================================================================

df = pd.DataFrame({
    'client_id': [f"CLT_{i:04d}" for i in range(N_SAMPLES)],
    'tech_literacy': client_tech_literacy,
    'ownership_clarity': ownership_clarity,
    'scope_lock_day': scope_lock_day,
    'asset_depth': asset_depth,
    'founder_involved': founder_involved,
    'repeat_query_rate': repeat_query_rate,
    'ttfv_days': ttfv_days.astype(int),
    'cx_satisfaction': cx_satisfaction
})

# Save the canonical dataset
df.to_csv('../data/onboarding_data_v2.csv', index=False)

print("SUCCESS: Canonical Dataset 'onboarding_data_v2.csv' generated in /data folder.")
print(df.head())

Initializing Data Engine: Generating 1000 observations...
SUCCESS: Canonical Dataset 'onboarding_data_v2.csv' generated in /data folder.
  client_id  tech_literacy  ownership_clarity  scope_lock_day  asset_depth  \
0  CLT_0000       0.423793           0.202579              11     0.386865   
1  CLT_0001       0.869333           0.453949              20     0.207787   
2  CLT_0002       0.646780           0.730470              11     0.616015   
3  CLT_0003       0.559571           0.592995              13     0.780537   
4  CLT_0004       0.263103           0.658817               7     0.559177   

   founder_involved  repeat_query_rate  ttfv_days  cx_satisfaction  
0                 1           0.308259         62         3.048296  
1                 0           0.055306         62         3.733590  
2                 0           0.189634         37         3.604397  
3                 1           0.143797         47         3.735201  
4                 1           0.349207         59